In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from src.preprocess import preprocess_data, NUMERIC_COLS


In [2]:
df = pd.read_csv("../data/raw/diabetic_data.csv")

X, y = preprocess_data(df)

print("Dataset shape:", X.shape)


Dataset shape: (101766, 47)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
#preprocessing
numeric_features = [col for col in NUMERIC_COLS if col in X.columns]

categorical_features = [
    col for col in X.columns if col not in numeric_features
]

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [5]:
#defining models
models = {

    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    )

}


In [6]:
#training and evaluating them
results = []

trained_models = {}

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:,1]

    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, probs)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc
    })

    trained_models[name] = pipeline


/Users/pradhyumnakasula/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [7]:
results_df = pd.DataFrame(results)

results_df.sort_values("ROC_AUC", ascending=False)


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
2,Gradient Boosting,0.888769,0.629630,0.007486,0.014795,0.677529
1,Random Forest,0.888425,0.000000,0.000000,0.000000,0.658411
0,Logistic Regression,0.644099,0.167625,0.552180,0.257178,0.643844


In [8]:
best_model_name = results_df.sort_values("ROC_AUC", ascending=False).iloc[0]["Model"]

best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

Best model: Gradient Boosting


In [9]:
best_preds = best_model.predict(X_test)

print(classification_report(y_test, best_preds))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.63      0.01      0.01      2271

    accuracy                           0.89     20354
   macro avg       0.76      0.50      0.48     20354
weighted avg       0.86      0.89      0.84     20354



In [10]:
import joblib

joblib.dump(best_model, "../models/best_model.pkl")

print("Model saved.")

Model saved.
